## Training
**Package:** `conditional_embedding_model`
**Source notebook:** `train/TrainNS_train.ipynb`
**Purpose:** Configure, instantiate, and train CenterEmbeddingResNetv4 using the package API, then inspect results.

## 1. Setup & Imports
All model and training logic comes from the `conditional_embedding_model` package.
`torch.nn`, standard libraries, and `matplotlib` are permitted for boilerplate and
visualization.

In [1]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader
from types import SimpleNamespace
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from conditional_embedding_model.models import CenterEmbeddingResNetv4
from conditional_embedding_model.training import (
    define_model,
    train_from_sweep,
    validate,
    plot_training_history,
    DynamicSigmoidBCELoss,
    set_seed,
    prediction_temperature,
    evaluate_model,
    evaluate_model_topk,
)
from conditional_embedding_model.data import load_data_grasp, batchify_wrapper

print("Imports OK")

Imports OK


pybullet build time: Jan 29 2025 23:16:28


## 2. Configuration
All tuneable values live in `CONFIG`. Change paths here — never in downstream cells.
`checkpoint_dir` is where the best-validation-loss checkpoint will be saved.

In [ ]:
CONFIG = {
    # paths
    "data_root":     "../config/data",
    "dataset_file":  "CoopGrasping-v6_dataset.pkl",
    "checkpoint_dir": "../config/weights",
    # model
    "input_size":    4,
    "embedding_size": 44,
    "dropout":       0.02,
    "model_version": "v4",
    # training
    "batch_size":             37,
    "learning_rate":          0.0002608572480321605,
    "lr_scheduler_patience":  3,
    "lr_scheduler_step":      0.374,
    "positive_weight":        1.0,
    "negative_weight":        1.0,
    "epochs":                 83,
    "max_length":             100,
    "multiplier_factor":      1.1,
    "temperature":            0.06154106204091678,
    "mixed_precision":        False,
    "weight_decay":           0.0001,
    "seed":                   1234,
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
model_name = (
    f"model_v{CONFIG['model_version']}"
    f"_e{CONFIG['epochs']}"
    f"_d{CONFIG['embedding_size']}"
    f"_lr{CONFIG['learning_rate']:.5f}.pth"
)
model_path = os.path.join(CONFIG["checkpoint_dir"], model_name)
print("Checkpoint path:", model_path)
print("Device:         ", CONFIG["device"])

## 3. Model Instantiation
`define_model` builds an `nn.Sequential(center_encoder, context_encoder)` pair.
Print the total trainable parameter count and the architecture string. If `torchinfo`
is available, also print a layer-by-layer summary for the center encoder.

In [ ]:
dataset_path = os.path.join(CONFIG["data_root"], CONFIG["dataset_file"])
train_dl, val_dl, test_ds, feature_struct = load_data_grasp(
    CONFIG["batch_size"],
    CONFIG["multiplier_factor"],
    dataset_path,
    CONFIG["max_length"],
)

net = define_model(
    input_size=CONFIG["input_size"],
    embedding_size=CONFIG["embedding_size"],
    feature_structure=feature_struct,
    dropout=CONFIG["dropout"],
    map_flag=True,
    device=CONFIG["device"],
    model_version=CONFIG["model_version"],
)

n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print()
print(net)

# %%
# torchinfo summary (optional)
try:
    from torchinfo import summary
    for batch in train_dl:
        c, ctx, msk, lbl, feat = [x.to(CONFIG["device"]) for x in batch]
        print("\n--- Center Encoder (net[0]) ---")
        summary(net[0], input_data=[c, feat], verbose=1)
        break
except ImportError:
    print("torchinfo not installed — skipping detailed layer summary")

## 4. Dataloader Setup
Dataloaders were already built in §3. This cell confirms the split sizes so the reader
can verify data is loaded correctly before starting training.

In [ ]:
print(f"Train batches  : {len(train_dl):>5d}")
print(f"Val batches    : {len(val_dl):>5d}")
print(f"Test samples   : {len(test_ds):>5d}")
print(f"Batch size     : {CONFIG['batch_size']:>5d}")
print(f"Max seq length : {CONFIG['max_length']:>5d}")

## 5. Training Run
`train_from_sweep` runs the full training loop: AdamW optimiser, ReduceLROnPlateau
scheduler, `DynamicSigmoidBCELoss`, and checkpoint saving. It also logs to Weights &
Biases — set your W&B entity/project in `trainer.py` or run `wandb login` before
executing this cell. After training, the returned history dict is plotted inline.

In [ ]:
set_seed(CONFIG["seed"])

# Build a SimpleNamespace so train_from_sweep can attribute-access the config
cfg_ns = SimpleNamespace(**CONFIG)
cfg_ns.dataset    = CONFIG["dataset_file"]
cfg_ns.map        = True
cfg_ns.model_path = model_path

net, history = train_from_sweep(
    net,
    train_dl,
    val_dl,
    cfg_ns,
    device=CONFIG["device"],
    model_path=model_path,
)
print("Training complete.")

Plot the training / validation loss curves recorded during training.
`plot_training_history` expects a dict with `train_loss` and `val_loss` lists.

In [ ]:
plot_training_history(history)

## 6. Sweep — optional, disabled by default
The cell below shows how to launch a Bayesian hyperparameter sweep over 350 runs.
It is wrapped in `if False:` so it never runs automatically.

To activate it:
1. Change `False` to `True`.
2. Ensure W&B credentials are configured (`wandb login`).
3. The sweep targets `test/pr_auc` (maximise) and searches `embedding_size`,
   `learning_rate`, `temperature`, `batch_size`, etc.

> **Warning:** `count=350` on a GPU cluster takes several hours.

In [ ]:
if False:  # Change to True to launch the sweep
    from conditional_embedding_model.training import sweep
    sweep()

## 7. Quick Post-Train Test
Run one forward pass on the test set and print the top-5 predictions alongside the
ground-truth positive indices for the first 8 samples. This is a smoke test — full
evaluation is in `03_evaluation.ipynb`.

In [ ]:
net.eval()
test_dl = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=batchify_wrapper(CONFIG["max_length"]),
)

for batch in test_dl:
    center, ctx_neg, mask, label, feature = [x.to(CONFIG["device"]) for x in batch]
    with torch.no_grad():
        logits = prediction_temperature(
            center, ctx_neg, net[0], net[1], feature,
            temperature=CONFIG["temperature"],
        )
        logits[mask == 0] = -float("inf")
        top5 = logits.topk(5, dim=1).indices  # (B, 5)

    print(f"{'Sample':>6}  {'Top-5 predictions':>28}  {'Ground-truth positives':>25}")
    print("-" * 65)
    for i in range(min(8, center.size(0))):
        pred_str = str(top5[i].cpu().tolist())
        true_idx = label[i].nonzero(as_tuple=True)[0][:5].tolist()
        print(f"{i:>6}  {pred_str:>28}  {str(true_idx):>25}")
    break